# 🔑 Solutions · Lesson 05: ARIMA, SARIMA & SARIMAX

> Try every problem yourself first! Peek at hints before peeking here. Run the setup cell, then read each solution **and its explanation**: there is often more than one valid way.

In [1]:
import sys; sys.path.append("..")            # lets us import course_utils.py from the parent folder
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.tools.sm_exceptions         # (imported first: statsmodels registers its own warning filters on import)
import warnings; warnings.filterwarnings("ignore")   # keep the notebook readable; in real projects READ your warnings!
from course_utils import *                    # data loaders, metrics, plotting helpers, check()

In [2]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
def build_exog(df, K=3):
    X = fourier_terms(df.index, K)
    X["rain6_l8"] = (df["rainfall_mm"].rolling(6).mean().shift(8) / 30)
    X["fuel_l8"] = np.log(df["fuel_index"] / 100).shift(8)
    X["fest"] = df["festival_flag"]
    return X

### ✅ Solution 5.1 · Identify and fit: tomato

**Problem recap:** Tomato prices are stationary in levels (Lesson 1). On the **log tomato price**, fit `ARIMA(order=(p,0,q))` for the candidates
  `(1,0,0)`, `(2,0,0)`, `(1,0,1)`, `(2,0,1)`, then choose the one with the lowest **AIC**. Store the winning order in `best_order` and print its AR coefficient(s).

In [3]:
from statsmodels.tsa.arima.model import ARIMA
lt = np.log(load_weekly('tomato')['price'])
aics = {o: ARIMA(lt, order=o).fit().aic for o in [(1, 0, 0), (2, 0, 0), (1, 0, 1), (2, 0, 1)]}
print({k: round(v, 1) for k, v in aics.items()})
best_order = min(aics, key=aics.get)
print('best order:', best_order)
print(ARIMA(lt, order=best_order).fit().params.round(3))

{(1, 0, 0): np.float64(-1162.4), (2, 0, 0): np.float64(-1167.0), (1, 0, 1): np.float64(-1167.0), (2, 0, 1): np.float64(-1165.0)}
best order: (2, 0, 0)
const     4.130
ar.L1     0.777
ar.L2     0.102
sigma2    0.009
dtype: float64


**Why this works:** Lower AIC = better trade-off between fit and complexity. Comparing AICs is legitimate here because all four models use the *same data and the same differencing* (d = 0). Never compare AIC across different values of d.

### ✅ Solution 5.2 · Residual diagnostics with Ljung–Box

**Problem recap:** For the best tomato model from 5.1, run the **Ljung–Box test** (`statsmodels.stats.diagnostic.acorr_ljungbox`) on the residuals at lags 10 and 26.
  The null hypothesis is *'residuals are white noise'*. Is it rejected? What would a rejection mean for your model?

In [4]:
from statsmodels.stats.diagnostic import acorr_ljungbox
res = ARIMA(lt, order=best_order).fit()
print(acorr_ljungbox(res.resid, lags=[10, 26]).round(4))

     lb_stat  lb_pvalue
10   26.3211     0.0033
26  103.3495     0.0000


**Why this works:** A small p-value says the residuals are *not* white noise: the model left predictable structure behind. For tomato, that leftover structure is the annual seasonality that a plain ARMA cannot represent, which motivates the seasonal terms or Fourier regressors we add next.

### ✅ Solution 5.3 · Grid-search a SARIMA on the airline data

**Problem recap:** On **log airline passengers** (hold out the last 24 months) fit `SARIMAX(order=(p,1,q), seasonal_order=(P,1,Q,12))` for every combination of
  `p, q, P, Q ∈ {0, 1}`. Pick the best by AIC. Forecast the 24 held-out months (remember to `np.exp` back!) and report RMSE next to the Holt-Winters
  multiplicative RMSE from Lesson 4 (≈ 41).

In [5]:
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX
air = load_airline()
train, test = air.iloc[:-24], air.iloc[-24:]
rows = []
for p, q, P, Q in itertools.product([0, 1], repeat=4):
    r = SARIMAX(np.log(train), order=(p, 1, q), seasonal_order=(P, 1, Q, 12)).fit(disp=False)
    rows.append(dict(p=p, q=q, P=P, Q=Q, AIC=r.aic))
grid = pd.DataFrame(rows).sort_values('AIC').reset_index(drop=True)
print(grid.head(4).round(1))
b = grid.iloc[0]
best = SARIMAX(np.log(train), order=(int(b.p), 1, int(b.q)), seasonal_order=(int(b.P), 1, int(b.Q), 12)).fit(disp=False)
fc = np.exp(best.forecast(24))
print('SARIMA RMSE:', round(rmse(test, fc), 1), '| Holt-Winters (mul) RMSE from Lesson 4: ~41 | seasonal naive:', round(rmse(test, fc_snaive(train, 24, m=12)), 1))

   p  q  P  Q    AIC
0  0  1  0  1 -389.0
1  1  0  0  1 -388.6
2  1  1  0  1 -387.3
3  0  1  1  1 -387.2
SARIMA RMSE: 43.2 | Holt-Winters (mul) RMSE from Lesson 4: ~41 | seasonal naive: 77.0


**Why this works:** The best order found is typically the famous 'airline model' SARIMA(0,1,1)(0,1,1)₁₂ (or one very close to it): one regular and one seasonal difference plus one regular and one seasonal MA term. Working on the **log** scale turns the multiplicative seasonality into an additive one that SARIMA can handle.

### ✅ Solution 5.4 · How many Fourier pairs?

**Problem recap:** For weekly **onion** log prices, fit `SARIMAX(order=(1,1,1), trend='n')` with Fourier terms `K = 1 … 6` as exogenous regressors (`fourier_terms(index, K)`)
  and choose `K` by **AIC** (and by **BIC**, which penalises complexity harder). Do they agree?

In [6]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
y = np.log(load_weekly('onion')['price'])
rows = {}
for K in range(1, 7):
    r = SARIMAX(y, exog=fourier_terms(y.index, K), order=(1, 1, 1), trend='n').fit(disp=False)
    rows[K] = {'AIC': r.aic, 'BIC': r.bic}
t = pd.DataFrame(rows).T
print(t.round(1))
print('best K by AIC:', t['AIC'].idxmin(), '| by BIC:', t['BIC'].idxmin())

      AIC     BIC
1 -2030.6 -2008.4
2 -2105.6 -2074.5
3 -2104.4 -2064.4
4 -2100.8 -2052.0
5 -2098.1 -2040.4
6 -2099.1 -2032.5
best K by AIC: 2 | by BIC: 2


**Why this works:** Each extra harmonic adds a finer seasonal wiggle. AIC/BIC trade the improved fit against the extra parameters; BIC usually picks a smaller K. Here both criteria choose K = 2: a couple of harmonics capture a smooth annual price cycle, and the AIC gets *worse* beyond that as extra harmonics mostly fit noise.

### ✅ Solution 5.5 · Do rainfall and fuel help for tomato?

**Problem recap:** Repeat the worked-example experiment for **tomato**: walk-forward at horizon 8 (`initial=470`, `step=8`) comparing
  (a) naive, (b) ARIMA(1,1,1) + Fourier (K=3), and (c) ARIMA(1,1,1) + Fourier + `rain6_l8` + `fuel_l8` + `fest`.
  Use the `build_exog` helper from the worked example. Does adding the exogenous drivers help? Why might the answer differ from maize?

In [7]:
tomato = load_weekly('tomato')
Xt = build_exog(tomato)
valid_t = Xt.dropna().index
def make_sarimax_t(cols):
    def f(train, h):
        idx = train.index.intersection(valid_t); tr = np.log(train.loc[idx])
        fut = pd.date_range(train.index[-1], periods=h + 1, freq='W-SUN')[1:]
        mod = SARIMAX(tr, exog=Xt.loc[idx, cols], order=(1, 1, 1), trend='n').fit(disp=False)
        return np.exp(mod.forecast(h, exog=Xt.loc[fut, cols]).values)
    return f
cf = [c for c in Xt.columns if c[:3] in ('sin', 'cos')]
ca = cf + ['rain6_l8', 'fuel_l8', 'fest']
out = {}
for name, f in {'naive': fc_naive, 'ARIMA + Fourier': make_sarimax_t(cf), 'ARIMA + Fourier + drivers': make_sarimax_t(ca)}.items():
    wf = walk_forward(tomato['price'], f, initial=470, horizon=8, step=8)
    out[name] = {'MAE': mae(wf.actual, wf.forecast), 'RMSE': rmse(wf.actual, wf.forecast)}
pd.DataFrame(out).T.round(2)

,MAE,RMSE
naive,8.31,10.29
ARIMA + Fourier,4.64,5.93
ARIMA + Fourier + drivers,4.33,5.54


**Why this works:** Seasonality gives the big win for tomato (RMSE roughly 10.3 → 5.9). The weather and fuel drivers add only a modest further gain (≈ 5.9 → 5.5, about 7 %), much less than for maize, where they were worth an extra ~20 %. Exogenous information helps in proportion to the signal it carries *relative to the noise*: tomato's weekly price noise is several times bigger than maize's, so the weather effect is harder to extract. Always let the walk-forward result decide rather than assuming 'more features = better'.